In [1]:
import os
import random
import shutil
import glob
from pathlib import Path
import pandas as pd
import numpy as np

from astropy.stats import sigma_clip
import astropy.io.ascii as ascii
import astropy.units as u

from spectres import spectres

from specutils import Spectrum1D
from specutils.manipulation import box_smooth
from specutils.spectra import Spectrum1D, SpectralRegion
from specutils.fitting import fit_generic_continuum

import torch
import visualtorch
import torch.nn as nn
import torch.nn.functional as F
from torch import nn
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor, Lambda
from torchmetrics.classification import BinaryConfusionMatrix
from torch.utils.data import Dataset, DataLoader, Subset, TensorDataset, Sampler
from torch.utils.tensorboard import SummaryWriter
from torcheval.metrics import BinaryAUROC
from torchviz import make_dot

2026-05-08 23:54:54.373286: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
torch.set_num_threads(1)

In [3]:
### DEFINE MODEL ###

device = torch.device("cpu")

class NovaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=2001, out_features=1000)
        self.layer_2 = nn.Linear(in_features=1000, out_features=1)

    def forward(self, x):
        x = F.relu(self.layer_1(x))
        x = self.layer_2(x)
        return x

model = NovaClassifier().to(device)
print(model)

NovaClassifier(
  (layer_1): Linear(in_features=2001, out_features=1000, bias=True)
  (layer_2): Linear(in_features=1000, out_features=1, bias=True)
)


In [9]:
### LOAD TRAINED MODEL ###

final_model = torch.load('final_model.pth')
final_model.eval()

NovaClassifier(
  (layer_1): Linear(in_features=2001, out_features=1000, bias=True)
  (layer_2): Linear(in_features=1000, out_features=1, bias=True)
)

In [13]:
path = 'your_folder' #Replace with your folder path

In [6]:
## Need to remove the factors making the flux values very small -- leads to numerical instability when continuum fitting 

def scale_flux(flux):
    scale_factors = [1.0E-21,1.0E-20,1.0E-19,1.0E-18,1.0E-17,1.0E-16,1.0E-15,1.0E-14,1.0E-13,1.0E-12,1.0E-11,1.0E-10]
    # for file in glob.glob(path):
    #     data = ascii.read(file, names=['wavelength','flux','error'])
    #     flux = data['flux']
    #     if max(flux) < 1.0E-3:
    #         print (file, max(flux))
    #         for i,F in enumerate(scale_factors[1:]):
    #             if max(flux) < F:
    #                 print (f"Use factor {scale_factors[i]}")
    #                 break

    if max(flux) < 1.0E-3:
        for i,F in enumerate(scale_factors[1:]):
            if max(flux) < F:
                return scale_factors[i]
    return 1.0

In [14]:
### PREPROCESS DATA AND PREDICT CLASS ###
regrid = np.arange(4000, 8002, 2)
resampled_arrays = []
predictions = []
filename_array = []

for file in glob.glob(path)[0:]:
    data = ascii.read(file, names=['wavelength','flux','error'])
    
    wavelength = data['wavelength']
    scale = scale_flux(data['flux'])
    flux = data['flux']/scale
    spectrum = Spectrum1D(flux=flux*(u.erg/(u.cm**2)/u.s/u.AA), spectral_axis=wavelength*u.AA)
    
    spec1_bsmooth = box_smooth(spectrum, width=6)


    # Clip points above a threshold (e.g., 3 sigma) to remove strong lines from the continuum fitting
    flux_masked = sigma_clip(flux, sigma=3, maxiters=2)
    mask = ~flux_masked.mask   # here clipped points set to False and others to be used for continuum fitting set to True
    g1_fit = fit_generic_continuum(Spectrum1D(flux=spectrum.flux[mask], spectral_axis=spectrum.spectral_axis[mask]))
    y_continuum_fitted = g1_fit(spectrum.spectral_axis)

    
    spec_normalized = spec1_bsmooth / y_continuum_fitted

    mm = np.ones_like(wavelength, dtype=bool)
    window = 10
    
    spec_resample, spec_errs_resample = spectres(regrid, wavelength[mm], spec_normalized.flux[mm], spec_errs=data['error'][mm]/scale)
            
    ## interpolate/extrapolate where resampled fluxes are NAN; not treating the resampled error bars though
    mask = np.isnan(spec_resample)
    spec_resample[mask] = np.interp(regrid[mask], regrid[~mask], spec_resample[~mask])

    ## ensuring the extrapolated regions are free of artifical edges when using the last value from original spectrum
    mask = regrid > max(wavelength)
    spec_resample[mask] = np.median(spec_normalized.flux[-window:])
    mask = regrid < min(wavelength)
    spec_resample[mask] = np.median(spec_normalized.flux[0:window])
    
    
    resampled_arrays.append(spec_resample)
    
    file_name = os.path.basename(file)
    filename_array.append(file_name)
        
    
    # Vertical normalization per sample (zero mean, unit variance)
    mean = np.mean(spec_resample)
    std = np.std(spec_resample)
    spec_resample = (spec_resample - mean) / std
    
    X = torch.tensor(spec_resample, dtype=torch.float32)
    
    #Use model to predict class
    with torch.no_grad():
        output = final_model(X)
        predictions.append(int(torch.round(torch.sigmoid(output)).item()))

In [15]:
predicted_dataset = pd.DataFrame({'filename': filename_array, 'prediction': predictions}, columns=['filename', 'prediction'])
print(predicted_dataset)

                           filename  prediction
0  AT2020fcw_2020_03_28_22_SNIa.txt           0
1       ZTF22abfxmpc_2022_11_01.txt           1
2       ZTF18acaqbgu_2019_09_21.txt           0
3     AT2019qml_2019_10_03_nova.txt           1
